# Dense Video Captioning — Full Local Pipeline
**Vid2Seq on YouCook2 | RTX 4060 Laptop GPU**

Full pipeline: video download → Whisper ASR → feature extraction → training → evaluation.
Project path: `D:\MS\UMD\Courses\Spring-2026\NLP`

## 1. Environment Check

In [3]:
import torch, sys, platform

print(f"Python  : {sys.version.split()[0]}")
print(f"OS      : {platform.system()} {platform.release()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: CUDA not available. Install CUDA-enabled PyTorch:")
    print("  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")


Python  : 3.11.7
OS      : Windows 10
PyTorch : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM    : 8.6 GB


## 2. Install Dependencies
Run once.

In [5]:
import subprocess, sys

packages = [
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121",
    "transformers==4.40.0",
    "sentencepiece",
    "openai-whisper",
    "h5py",
    "evaluate",
    "nltk",
    "huggingface_hub",
    "tqdm",
    "opencv-python",
    "Pillow",
    "numpy",
    "pycocoevalcap",
]

for pkg in packages:
    print(f"Installing {pkg.split()[0]}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkg.split(), check=False)

print("\nDone.")


Installing torch...
Installing transformers==4.40.0...
Installing sentencepiece...
Installing openai-whisper...
Installing h5py...
Installing evaluate...
Installing nltk...
Installing huggingface_hub...
Installing tqdm...
Installing opencv-python...
Installing Pillow...
Installing numpy...
Installing pycocoevalcap...

Done.


## 3. Project Paths

In [21]:
import os

PROJECT_ROOT = r"D:\MS\UMD\Courses\Spring-2026\NLP"

PATHS = {
    "raw_videos"     : os.path.join(PROJECT_ROOT, "data", "raw_videos"),
    "annotations"    : os.path.join(PROJECT_ROOT, "data", "annotations"),
    "asr_transcripts": os.path.join(PROJECT_ROOT, "data", "asr_transcripts"),
    "features"       : os.path.join(PROJECT_ROOT, "data", "features"),
    "checkpoints"    : os.path.join(PROJECT_ROOT, "checkpoints"),
    "outputs"        : os.path.join(PROJECT_ROOT, "outputs"),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  {name:20s}: {path}")

print("\nAll directories ready.")


  raw_videos          : D:\MS\UMD\Courses\Spring-2026\NLP\data\raw_videos
  annotations         : D:\MS\UMD\Courses\Spring-2026\NLP\data\annotations
  asr_transcripts     : D:\MS\UMD\Courses\Spring-2026\NLP\data\asr_transcripts
  features            : D:\MS\UMD\Courses\Spring-2026\NLP\data\features
  checkpoints         : D:\MS\UMD\Courses\Spring-2026\NLP\checkpoints
  outputs             : D:\MS\UMD\Courses\Spring-2026\NLP\outputs

All directories ready.


## 4. Download YouCook2 Annotations

In [24]:
import urllib.request, os

ANNOTATIONS_DIR = PATHS["annotations"]

URLS = {
    "youcookii_annotations_trainval.json":
        "http://youcook2.eecs.umich.edu/static/YouCookII/youcookii_annotations_trainval.json",
    "youcookii_annotations_test_segments_only.json":
        "http://youcook2.eecs.umich.edu/static/YouCookII/youcookii_annotations_test_segments_only.json",
}

for filename, url in URLS.items():
    dest = os.path.join(ANNOTATIONS_DIR, filename)
    if os.path.exists(dest):
        print(f"  Already exists: {filename}  ({os.path.getsize(dest)//1024} KB)")
        continue
    print(f"  Downloading {filename} ...")
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"  Saved ({os.path.getsize(dest)//1024} KB)")
    except Exception as e:
        print(f"  FAILED: {e}")
        print(f"  Manual URL: {url}")


  Already exists: youcookii_annotations_trainval.json  (1538 KB)
  Already exists: youcookii_annotations_test_segments_only.json  (109 KB)


## 5. Load & Explore Annotations

In [27]:
import json, os
from collections import Counter

ANNOTATIONS_DIR = PATHS["annotations"]

with open(os.path.join(ANNOTATIONS_DIR, "youcookii_annotations_trainval.json")) as f:
    trainval_data = json.load(f)
with open(os.path.join(ANNOTATIONS_DIR, "youcookii_annotations_test_segments_only.json")) as f:
    test_data = json.load(f)

db = trainval_data["database"]

subsets   = Counter(v["subset"] for v in db.values())
recipes   = Counter(v["recipe_type"] for v in db.values())
seg_counts = [len(v["annotations"]) for v in db.values()]
durations  = [v["duration"] for v in db.values() if v.get("duration")]

print("Splits          :", dict(subsets))
print("Recipe types    :", len(recipes))
print(f"Segments/video  : min={min(seg_counts)}, max={max(seg_counts)}, avg={sum(seg_counts)/len(seg_counts):.1f}")
print(f"Duration (s)    : min={min(durations):.0f}, max={max(durations):.0f}, avg={sum(durations)/len(durations):.0f}")
print(f"Total segments  : {sum(seg_counts)}")

sample_id = list(db.keys())[0]
s = db[sample_id]
print(f"\nSample video : {sample_id}")
print(f"  Recipe     : {s['recipe_type']}")
print(f"  Subset     : {s['subset']}")
print(f"  Segments   : {len(s['annotations'])}")
print(f"  First ann  : {s['annotations'][0]}")


Splits          : {'training': 1333, 'validation': 457}
Recipe types    : 89
Segments/video  : min=3, max=16, avg=7.7
Duration (s)    : min=44, max=1106, avg=315
Total segments  : 13829

Sample video : GLd3aX16zBg
  Recipe     : 113
  Subset     : training
  Segments   : 5
  First ann  : {'segment': [90, 102], 'id': 0, 'sentence': 'spread margarine on two slices of white bread'}


## 6. Sample Working Subset
50 train + 20 val videos.

In [30]:
import random, json, os

random.seed(42)

train_ids = [vid for vid, info in db.items() if info["subset"] == "training"]
val_ids   = [vid for vid, info in db.items() if info["subset"] == "validation"]

sample_train = random.sample(train_ids, 50)
sample_val   = random.sample(val_ids, 20)

subset = {
    "train": {vid: db[vid] for vid in sample_train},
    "val"  : {vid: db[vid] for vid in sample_val},
}

subset_path = os.path.join(PATHS["annotations"], "subset_annotations.json")
with open(subset_path, "w") as f:
    json.dump(subset, f, indent=2)

total_segs = (sum(len(v["annotations"]) for v in subset["train"].values()) +
              sum(len(v["annotations"]) for v in subset["val"].values()))

print(f"Train videos : {len(subset['train'])}")
print(f"Val videos   : {len(subset['val'])}")
print(f"Total segs   : {total_segs}")
print(f"Saved to     : {subset_path}")


Train videos : 50
Val videos   : 20
Total segs   : 506
Saved to     : D:\MS\UMD\Courses\Spring-2026\NLP\data\annotations\subset_annotations.json


## 7. Download Videos with yt-dlp
Node.js v22 is installed — we pass `--js-runtimes node` explicitly to suppress the warning.

In [33]:
import subprocess, os
from tqdm import tqdm

RAW_VIDEOS_DIR = PATHS["raw_videos"]

def download_video(youtube_id, url, output_dir):
    output_path = os.path.join(output_dir, f"{youtube_id}.mp4")
    if os.path.exists(output_path) and os.path.getsize(output_path) > 10_000:
        return "already_exists"

    cmd = [
        "yt-dlp",
        "--js-runtimes", "node",          # use Node.js v22 explicitly
        "-f", "bestvideo[height<=360][ext=mp4]+bestaudio[ext=m4a]/best[height<=360][ext=mp4]/best[height<=360]",
        "--merge-output-format", "mp4",
        "-o", output_path,
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        url,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    if result.returncode == 0 and os.path.exists(output_path):
        return "success"
    else:
        return f"failed: {result.stderr[:120]}"

# ── Test with 3 videos first ──────────────────────────────────────────────────
print("Testing with 3 videos...")
test_ids = list(subset["train"].keys())[:3]
for vid_id in test_ids:
    url = db[vid_id]["video_url"]
    status = download_video(vid_id, url, RAW_VIDEOS_DIR)
    print(f"  {vid_id}: {status}")

files = [f for f in os.listdir(RAW_VIDEOS_DIR) if f.endswith(".mp4")]
print(f"\nVideos downloaded so far: {len(files)}")


Testing with 3 videos...
  H3cnPVPdwTY: success
  k-aTILi_nLY: success
  PKt_za_XfF8: success

Videos downloaded so far: 3


In [34]:
# ── Download all 70 videos ────────────────────────────────────────────────────
from tqdm import tqdm

all_ids = list(subset["train"].keys()) + list(subset["val"].keys())
results = {"success": 0, "already_exists": 0, "failed": []}

for vid_id in tqdm(all_ids, desc="Downloading"):
    url    = db[vid_id]["video_url"]
    status = download_video(vid_id, url, RAW_VIDEOS_DIR)
    if status == "success":
        results["success"] += 1
    elif status == "already_exists":
        results["already_exists"] += 1
    else:
        results["failed"].append((vid_id, status))

print(f"\nSuccess       : {results['success']}")
print(f"Already existed: {results['already_exists']}")
print(f"Failed         : {len(results['failed'])}")
for vid_id, reason in results["failed"]:
    print(f"  {vid_id}: {reason}")

files = [f for f in os.listdir(RAW_VIDEOS_DIR) if f.endswith(".mp4")]
print(f"\nTotal videos in raw_videos: {len(files)}")


Downloading: 100%|█████████████████████████████████████████████████████████████████████| 70/70 [07:46<00:00,  6.67s/it]


Success       : 56
Already existed: 3
Failed         : 11
  FHvZgt3ExDI: failed: ERROR: [generic] 'cc' is not a valid URL

  SVihYubF078: failed: ERROR: [youtube] SVihYubF078: Video unavailable

  igEW1p4ZViM: failed: ERROR: [youtube] igEW1p4ZViM: Private video. Sign in if you've been granted access to this video. Use --cookies-from-bro
  s345cUBQhhk: failed: ERROR: [youtube] s345cUBQhhk: Private video. Sign in if you've been granted access to this video. Use --cookies-from-bro
  91Fz5ZBgeL4: failed: ERROR: [youtube] 91Fz5ZBgeL4: Video unavailable

  JYzdSgVKWTA: failed: ERROR: [youtube] JYzdSgVKWTA: Private video. Sign in if you've been granted access to this video. Use --cookies-from-bro
  zuQfLg46-Yc: failed: ERROR: [youtube] zuQfLg46-Yc: Private video. Sign in if you've been granted access to this video. Use --cookies-from-bro
  xAbBPXAnYvw: failed: ERROR: [youtube] xAbBPXAnYvw: Video unavailable. This video is no longer available because the YouTube account associate
  yreC9D4yYi

## 8. Filter Subset to Successfully Downloaded Videos

In [36]:
import os, json

downloaded = set(
    f.replace(".mp4", "")
    for f in os.listdir(PATHS["raw_videos"])
    if f.endswith(".mp4") and os.path.getsize(os.path.join(PATHS["raw_videos"], f)) > 10_000
)

clean_subset = {
    "train": {vid: data for vid, data in subset["train"].items() if vid in downloaded},
    "val"  : {vid: data for vid, data in subset["val"].items()   if vid in downloaded},
}

total_segs = (sum(len(v["annotations"]) for v in clean_subset["train"].values()) +
              sum(len(v["annotations"]) for v in clean_subset["val"].values()))

print(f"Clean train : {len(clean_subset['train'])} videos")
print(f"Clean val   : {len(clean_subset['val'])} videos")
print(f"Total segs  : {total_segs}")

subset_path = os.path.join(PATHS["annotations"], "subset_annotations.json")
with open(subset_path, "w") as f:
    json.dump(clean_subset, f, indent=2)
print(f"Saved: {subset_path}")


Clean train : 42 videos
Clean val   : 17 videos
Total segs  : 432
Saved: D:\MS\UMD\Courses\Spring-2026\NLP\data\annotations\subset_annotations.json


## 9. Whisper ASR Transcription
Whisper-medium on your RTX 4060. ~8-12s per video.

In [38]:
import whisper, subprocess, json, os, torch
from tqdm import tqdm

ASR_DIR       = PATHS["asr_transcripts"]
RAW_VIDEOS_DIR = PATHS["raw_videos"]

device = "cuda" if torch.cuda.is_available() else "cpu"
model_whisper = whisper.load_model("medium", device=device)
print(f"Whisper medium loaded on {device}")

all_ids = list(clean_subset["train"].keys()) + list(clean_subset["val"].keys())
failed  = []

for vid_id in tqdm(all_ids, desc="Transcribing"):
    out_path = os.path.join(ASR_DIR, f"{vid_id}.json")
    if os.path.exists(out_path):
        continue

    video_path = os.path.join(RAW_VIDEOS_DIR, f"{vid_id}.mp4")
    audio_path = os.path.join(ASR_DIR, f"{vid_id}_tmp.wav")

    # Extract audio with ffmpeg
    cmd = ["ffmpeg", "-i", video_path, "-ar", "16000", "-ac", "1",
           "-q:a", "0", audio_path, "-y", "-loglevel", "error"]
    res = subprocess.run(cmd, capture_output=True)
    if res.returncode != 0:
        failed.append(vid_id)
        continue

    result = model_whisper.transcribe(audio_path, verbose=False)
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)
    os.remove(audio_path)

print(f"\nTranscribed : {len(all_ids) - len(failed)}")
print(f"Failed      : {len(failed)}")
if failed:
    print("Failed IDs:", failed)


100%|█████████████████████████████████████| 1.42G/1.42G [00:48<00:00, 31.2MiB/s]


Whisper medium loaded on cuda


Transcribing:   0%|                                                                             | 0/59 [00:00<?, ?it/s]

Detected language: English



%|                                                                                    | 0/58967 [00:00<?, ?frames/s]
%|███▏                                                                    | 2576/58967 [00:08<03:04, 305.00frames/s]
%|██████▌                                                                 | 5400/58967 [00:16<02:43, 327.66frames/s]
%|█████████▎                                                              | 7616/58967 [00:22<02:31, 339.40frames/s]
%|████████████▍                                                          | 10280/58967 [00:28<02:10, 372.67frames/s]
%|███████████████▌                                                       | 12944/58967 [00:36<02:07, 362.25frames/s]
%|██████████████████▍                                                    | 15312/58967 [00:44<02:09, 336.03frames/s]
%|█████████████████████▋                                                 | 17976/58967 [00:51<01:53, 359.92frames/s]
%|████████████████████████                                     

...

Detected language: English



%|                                                                                    | 0/29965 [00:00<?, ?frames/s]
%|██████▉                                                                 | 2884/29965 [00:10<01:37, 277.91frames/s]
%|████████████                                                            | 5032/29965 [00:17<01:27, 286.50frames/s]
%|█████████████████▌                                                      | 7300/29965 [00:24<01:15, 299.87frames/s]
%|███████████████████████▌                                                | 9792/29965 [00:32<01:04, 314.01frames/s]
%|███████████████████████████▎                                           | 11506/29965 [00:36<00:54, 338.24frames/s]
%|█████████████████████████████████▋                                     | 14218/29965 [00:42<00:43, 362.45frames/s]
%|███████████████████████████████████████▋                               | 16754/29965 [00:51<00:39, 332.33frames/s]
%|█████████████████████████████████████████████▋               

Detected language: French



%|                                                                                    | 0/55481 [00:00<?, ?frames/s]
%|███▋                                                                    | 2860/55481 [00:05<01:35, 553.26frames/s]
%|███████▏                                                                | 5540/55481 [00:11<01:49, 454.13frames/s]
%|██████████▍                                                             | 8040/55481 [00:19<01:58, 402.04frames/s]
%|█████████████▏                                                         | 10340/55481 [00:21<01:28, 507.69frames/s]
%|████████████████▊                                                      | 13140/55481 [00:24<01:08, 615.21frames/s]
%|████████████████████▎                                                  | 15840/55481 [00:30<01:14, 534.64frames/s]
%|███████████████████████▉                                               | 18740/55481 [00:34<01:01, 596.90frames/s]
%|███████████████████████████▍                                 


Transcribed : 59
Failed      : 0


In [39]:
# Verify transcripts
import os, json

ASR_DIR = PATHS["asr_transcripts"]
files   = [f for f in os.listdir(ASR_DIR) if f.endswith(".json")]
print(f"Total transcripts: {len(files)}")

if files:
    sample_id = files[0].replace(".json", "")
    with open(os.path.join(ASR_DIR, f"{sample_id}.json")) as f:
        t = json.load(f)
    print(f"\nSample: {sample_id}")
    print(f"Language : {t.get('language')}")
    print(f"Preview  : {t['text'][:200]}")
    for seg in t.get("segments", [])[:2]:
        print(f"  [{seg['start']:.1f}s-{seg['end']:.1f}s]: {seg['text'].strip()}")


Total transcripts: 59

Sample: 0eCScMSsl_Y
Language : en
Preview  :  wondering what to make for dinner tonight we have some quick and easy tips for serving up a restaurant quality meal the whole family will love today we're making a classic chicken parmesan you're gon
  [0.0s-7.1s]: wondering what to make for dinner tonight we have some quick and easy tips
  [7.1s-11.2s]: for serving up a restaurant quality meal the whole family will love today we're


## 10. Visual Feature Extraction with S3D
S3D pretrained on HowTo100M — same as Vid2Seq paper. Downloads ~100MB model weights once.

In [41]:
# Install torchvision S3D dependency
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "pytorchvideo"], check=False)

import torch
import torch.nn as nn
from torchvision.models.video import s3d, S3D_Weights
import torchvision.transforms as T
import cv2, h5py, os, numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load S3D with pretrained Kinetics-400 weights (closest publicly available)
# Note: HowTo100M weights require manual download; Kinetics weights work well
weights = S3D_Weights.DEFAULT
s3d_model = s3d(weights=weights)
# Remove classification head — keep feature trunk only
feature_extractor = nn.Sequential(*list(s3d_model.children())[:-1])
feature_extractor = feature_extractor.to(device).eval()

# S3D expects clips of 16 frames at 224x224
CLIP_LEN   = 16
FRAME_SIZE = 224
FEAT_DIM   = 1024   # S3D output dim

transform = weights.transforms()  # standard S3D preprocessing

def extract_s3d_features(video_path, fps_sample=1):
    """
    Sample clips at fps_sample Hz, extract S3D features.
    Returns features (N, 1024) and timestamps (N,).
    """
    cap  = cv2.VideoCapture(video_path)
    fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Sample center frame every 1/fps_sample seconds
    step = max(1, int(fps / fps_sample))
    sample_frames = list(range(0, total_frames, step))

    features, timestamps = [], []

    for center in sample_frames:
        # Grab CLIP_LEN frames centered on this position
        start = max(0, center - CLIP_LEN // 2)
        frames = []
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        for _ in range(CLIP_LEN):
            ret, frame = cap.read()
            if not ret:
                if frames:
                    frames.append(frames[-1])   # repeat last frame
                else:
                    frames.append(np.zeros((FRAME_SIZE, FRAME_SIZE, 3), dtype=np.uint8))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (FRAME_SIZE, FRAME_SIZE))
                frames.append(frame)

        # Stack to (C, T, H, W) float tensor
        clip = np.stack(frames, axis=0)               # (T, H, W, C)
        clip = torch.from_numpy(clip).permute(3, 0, 1, 2).float() / 255.0
        clip = clip.unsqueeze(0).to(device)           # (1, C, T, H, W)

        with torch.no_grad():
            feat = feature_extractor(clip)            # (1, 1024, 1, 1, 1) or similar
            feat = feat.squeeze().cpu().numpy()

        if feat.ndim == 0:
            feat = feat.reshape(1)
        features.append(feat.flatten()[:FEAT_DIM])
        timestamps.append(center / fps)

    cap.release()
    return np.array(features, dtype=np.float32), np.array(timestamps, dtype=np.float32)

# Test on one video
sample_vid = list(clean_subset["train"].keys())[0]
vid_path   = os.path.join(PATHS["raw_videos"], f"{sample_vid}.mp4")
feats, times = extract_s3d_features(vid_path)
print(f"Sample video  : {sample_vid}")
print(f"Feature shape : {feats.shape}  (frames × {FEAT_DIM})")
print(f"Duration      : {times[-1]:.1f}s")


Device: cuda


Downloading: "https://download.pytorch.org/models/s3d-d76dad2f.pth" to C:\Users\amans/.cache\torch\hub\checkpoints\s3d-d76dad2f.pth
100%|█████████████████████████████████████████████████████████████████████████████| 32.0M/32.0M [00:01<00:00, 20.2MB/s]


Sample video  : H3cnPVPdwTY
Feature shape : (610, 1024)  (frames × 1024)
Duration      : 589.3s


In [42]:
# Extract features for all videos → save to HDF5
import h5py, os
from tqdm import tqdm

H5_PATH = os.path.join(PATHS["features"], "video_features_s3d.h5")
all_ids = list(clean_subset["train"].keys()) + list(clean_subset["val"].keys())
failed  = []

with h5py.File(H5_PATH, "a") as hf:   # "a" = append mode, safe to resume
    for vid_id in tqdm(all_ids, desc="Extracting S3D features"):
        if vid_id in hf:               # already done
            continue
        vid_path = os.path.join(PATHS["raw_videos"], f"{vid_id}.mp4")
        if not os.path.exists(vid_path):
            failed.append(vid_id)
            continue
        try:
            feats, times = extract_s3d_features(vid_path)
            grp = hf.create_group(vid_id)
            grp.create_dataset("features",   data=feats,  compression="gzip")
            grp.create_dataset("timestamps", data=times)
        except Exception as e:
            failed.append(vid_id)
            print(f"  {vid_id}: {e}")

print(f"\nExtracted : {len(all_ids) - len(failed)} videos")
print(f"Failed    : {len(failed)}")

# Verify
with h5py.File(H5_PATH, "r") as hf:
    print(f"Videos in HDF5 : {len(hf.keys())}")
    sample = list(hf.keys())[0]
    print(f"Sample shape   : {hf[sample]['features'].shape}")


Extracting S3D features: 100%|█████████████████████████████████████████████████████████| 59/59 [16:38<00:00, 16.93s/it]


Extracted : 59 videos
Failed    : 0
Videos in HDF5 : 59
Sample shape   : (152, 1024)


## 11. PyTorch Dataset

In [44]:
import torch, h5py, json, os
import numpy as np
from torch.utils.data import Dataset, DataLoader

H5_PATH = os.path.join(PATHS["features"], "video_features_s3d.h5")
FEAT_DIM     = 1024
MAX_FEAT_LEN = 128

class DVCDataset(Dataset):
    """
    One sample = one annotated segment.
    Returns visual features (padded to MAX_FEAT_LEN),
    ASR text, ground-truth caption, and segment timestamps.
    """
    def __init__(self, subset_dict, h5_path, asr_dir, max_feat_len=MAX_FEAT_LEN):
        self.samples      = []
        self.max_feat_len = max_feat_len

        with h5py.File(h5_path, "r") as hf:
            available = set(hf.keys())
            for vid_id, info in subset_dict.items():
                if vid_id not in available:
                    continue

                feats = hf[vid_id]["features"][:]     # (T, 1024)
                times = hf[vid_id]["timestamps"][:]   # (T,)

                asr_path = os.path.join(asr_dir, f"{vid_id}.json")
                asr_text = ""
                if os.path.exists(asr_path):
                    try:
                        with open(asr_path) as f:
                            asr_text = json.load(f)["text"][:512]
                    except Exception:
                        pass

                for ann in info.get("annotations", []):
                    start, end = ann["segment"]
                    caption    = ann["sentence"]

                    mask      = (times >= start) & (times <= end)
                    seg_feats = feats[mask]

                    if len(seg_feats) == 0:
                        mid = (start + end) / 2
                        idx = int(np.argmin(np.abs(times - mid)))
                        seg_feats = feats[max(0, idx-2): idx+3]

                    if len(seg_feats) == 0:
                        continue

                    # Pad / truncate
                    if len(seg_feats) > max_feat_len:
                        seg_feats = seg_feats[:max_feat_len]
                    else:
                        pad = np.zeros((max_feat_len - len(seg_feats), FEAT_DIM), dtype=np.float32)
                        seg_feats = np.vstack([seg_feats, pad])

                    self.samples.append({
                        "video_id": vid_id,
                        "features": torch.tensor(seg_feats, dtype=torch.float32),
                        "asr_text": asr_text,
                        "caption" : caption,
                        "segment" : (start, end),
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


train_dataset = DVCDataset(clean_subset["train"], H5_PATH, PATHS["asr_transcripts"])
val_dataset   = DVCDataset(clean_subset["val"],   H5_PATH, PATHS["asr_transcripts"])

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")

s = train_dataset[0]
print(f"\nSample:")
print(f"  video_id : {s['video_id']}")
print(f"  features : {s['features'].shape}")
print(f"  caption  : {s['caption']}")
print(f"  segment  : {s['segment']}")
print(f"  ASR      : {'yes' if s['asr_text'] else 'no'}")


Train samples : 297
Val samples   : 135

Sample:
  video_id : H3cnPVPdwTY
  features : torch.Size([128, 1024])
  caption  : add soy sauce water sesame seed oil rice wine and brown sugar to a bowl and stir
  segment  : (203, 256)
  ASR      : yes


## 12. DataLoaders & Tokenizer

In [46]:
from torch.utils.data import DataLoader
from transformers import T5Tokenizer
import torch

tokenizer = T5Tokenizer.from_pretrained("t5-base")

# Add 100 timestamp tokens to vocabulary
N_TS = 100
timestamp_tokens = [f"<t_{i}>" for i in range(N_TS)]
tokenizer.add_tokens(timestamp_tokens)
print(f"Vocab size with timestamp tokens: {len(tokenizer)}")

def collate_fn(batch):
    features  = torch.stack([b["features"] for b in batch])
    captions  = [b["caption"]  for b in batch]
    asr_texts = [b["asr_text"] for b in batch]

    # Tokenize captions as decoder labels
    label_enc = tokenizer(captions, padding=True, truncation=True,
                           max_length=64, return_tensors="pt")
    labels = label_enc["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100   # ignore pad in loss

    # Tokenize ASR as auxiliary encoder input
    asr_enc = tokenizer(asr_texts, padding=True, truncation=True,
                         max_length=128, return_tensors="pt")
    return {
        "features"     : features,
        "labels"       : labels,
        "asr_input_ids": asr_enc["input_ids"],
        "asr_attn_mask": asr_enc["attention_mask"],
        "captions"     : captions,
    }

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                           collate_fn=collate_fn, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False,
                           collate_fn=collate_fn, num_workers=0, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  features      : {batch['features'].shape}")
print(f"  labels        : {batch['labels'].shape}")
print(f"  asr_input_ids : {batch['asr_input_ids'].shape}")


D:\Applications\Anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Vocab size with timestamp tokens: 32200
Train batches : 75
Val batches   : 34

Sample batch:
  features      : torch.Size([4, 128, 1024])
  labels        : torch.Size([4, 15])
  asr_input_ids : torch.Size([4, 128])


## 13. Vid2Seq-Style Model
T5-Base with visual projection + ASR fusion + timestamp tokens.

In [51]:
import torch
import torch.nn as nn
from transformers import T5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

class Vid2SeqLocal(nn.Module):
    """
    Vid2Seq-inspired model:
      - visual_proj  : Linear(1024 → 512) + LayerNorm + Dropout
      - t5           : T5-Base seq2seq decoder
      - ASR tokens fused by prepending to visual encoder states
    """
    def __init__(self, feat_dim=FEAT_DIM, hidden_dim=768):
        super().__init__()
        self.visual_proj = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.1),
        )
        self.t5 = T5ForConditionalGeneration.from_pretrained("t5-base")
        self.t5.resize_token_embeddings(len(tokenizer))   # add timestamp tokens

    def _encode(self, visual_feats, asr_input_ids, asr_attn_mask):
        vis = self.visual_proj(visual_feats)               # (B, T, 512)
        if asr_input_ids is not None:
            asr_hidden = self.t5.encoder(
                input_ids=asr_input_ids,
                attention_mask=asr_attn_mask,
            ).last_hidden_state                            # (B, L, 512)
            combined = torch.cat([vis, asr_hidden], dim=1)
        else:
            combined = vis
        return combined

    def forward(self, visual_feats, labels=None,
                asr_input_ids=None, asr_attn_mask=None):
        enc_hidden = self._encode(visual_feats, asr_input_ids, asr_attn_mask)
        return self.t5(
            encoder_outputs=BaseModelOutput(last_hidden_state=enc_hidden),
            labels=labels,
        )

    @torch.no_grad()
    def generate_caption(self, visual_feats,
                          asr_input_ids=None, asr_attn_mask=None,
                          max_length=80):
        enc_hidden = self._encode(visual_feats, asr_input_ids, asr_attn_mask)
        ids = self.t5.generate(
            encoder_outputs=BaseModelOutput(last_hidden_state=enc_hidden),
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
        )
        return tokenizer.batch_decode(ids, skip_special_tokens=True)


model = Vid2SeqLocal().to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total/1e6:.1f}M")
print(f"Trainable params : {trainable/1e6:.1f}M")

# Quick forward-pass smoke test
b = next(iter(train_loader))
out = model(
    b["features"].to(device),
    labels=b["labels"].to(device),
    asr_input_ids=b["asr_input_ids"].to(device),
    asr_attn_mask=b["asr_attn_mask"].to(device),
)
print(f"Smoke test loss  : {out.loss.item():.4f}  ✅")


Training on: cuda
Total params     : 223.7M
Trainable params : 223.7M
Smoke test loss  : 22.3086  ✅


## 14. Training
fp16 + gradient accumulation. ~25-35 min/epoch on RTX 4060.

In [54]:
import torch, os, json, time
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from tqdm import tqdm

# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_EPOCHS        = 10
LR                = 3e-4
WARMUP_STEPS      = 100
GRAD_ACCUM        = 8       # effective batch = 4 × 8 = 32
MAX_GRAD_NORM     = 1.0
SAVE_EVERY        = 2       # save checkpoint every N epochs
# ─────────────────────────────────────────────────────────────────────────────

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_STEPS)
scaler    = GradScaler()

history   = {"train_loss": [], "val_loss": []}
global_step = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    t_loss, n = 0.0, 0
    optimizer.zero_grad()
    t0 = time.time()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Ep {epoch}/{NUM_EPOCHS} train")):
        feats    = batch["features"].to(device)
        labels   = batch["labels"].to(device)
        asr_ids  = batch["asr_input_ids"].to(device)
        asr_mask = batch["asr_attn_mask"].to(device)

        with autocast():
            loss = model(feats, labels=labels,
                         asr_input_ids=asr_ids,
                         asr_attn_mask=asr_mask).loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            global_step += 1
            if global_step <= WARMUP_STEPS:
                scheduler.step()

        t_loss += loss.item() * GRAD_ACCUM
        n      += 1

    avg_train = t_loss / n

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Ep {epoch}/{NUM_EPOCHS} val  "):
            feats    = batch["features"].to(device)
            labels   = batch["labels"].to(device)
            asr_ids  = batch["asr_input_ids"].to(device)
            asr_mask = batch["asr_attn_mask"].to(device)
            with autocast():
                v_loss += model(feats, labels=labels,
                                asr_input_ids=asr_ids,
                                asr_attn_mask=asr_mask).loss.item()
            v += 1

    avg_val  = v_loss / max(v, 1)
    elapsed  = (time.time() - t0) / 60

    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)

    print(f"Epoch {epoch:2d} | train={avg_train:.4f} | val={avg_val:.4f} | {elapsed:.1f} min")

    # ── Checkpoint ────────────────────────────────────────────────────────────
    if epoch % SAVE_EVERY == 0:
        ckpt = os.path.join(PATHS["checkpoints"], f"epoch_{epoch:02d}.pt")
        torch.save({
            "epoch"      : epoch,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "train_loss" : avg_train,
            "val_loss"   : avg_val,
        }, ckpt)
        print(f"  → saved {ckpt}")

with open(os.path.join(PATHS["outputs"], "history.json"), "w") as f:
    json.dump(history, f, indent=2)
print("\nTraining complete.")


C:\Users\amans\AppData\Local\Temp\ipykernel_49332\3206991610.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
C:\Users\amans\AppData\Local\Temp\ipykernel_49332\3206991610.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
D:\Applications\Anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Ep 1/10 train: 100%|████████████████████████████████████████████████████████████████

Epoch  1 | train=18.7486 | val=11.5082 | 0.9 min


Ep 2/10 val  : 100%|███████████████████████████████████████████████████████████████████| 34/34 [00:05<00:00,  6.02it/s]


Epoch  2 | train=8.6465 | val=6.4340 | 1.0 min
  → saved D:\MS\UMD\Courses\Spring-2026\NLP\checkpoints\epoch_02.pt


Ep 3/10 val  : 100%|███████████████████████████████████████████████████████████████████| 34/34 [00:05<00:00,  6.03it/s]


...

Epoch  8 | train=3.8497 | val=3.9299 | 1.0 min
  → saved D:\MS\UMD\Courses\Spring-2026\NLP\checkpoints\epoch_08.pt


Ep 9/10 val  : 100%|███████████████████████████████████████████████████████████████████| 34/34 [00:05<00:00,  6.00it/s]


Epoch  9 | train=nan | val=3.7535 | 1.0 min


Ep 10/10 val  : 100%|██████████████████████████████████████████████████████████████████| 34/34 [00:05<00:00,  6.02it/s]


Epoch 10 | train=nan | val=3.6306 | 1.0 min
  → saved D:\MS\UMD\Courses\Spring-2026\NLP\checkpoints\epoch_10.pt

Training complete.


## 15. Loss Curve

In [56]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json, os

with open(os.path.join(PATHS["outputs"], "history.json")) as f:
    h = json.load(f)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(h["train_loss"], marker="o", label="Train")
ax.plot(h["val_loss"],   marker="s", label="Val")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training Progress — Dense Video Captioning")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()

out = os.path.join(PATHS["outputs"], "loss_curve.png")
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")


Saved: D:\MS\UMD\Courses\Spring-2026\NLP\outputs\loss_curve.png


C:\Users\amans\AppData\Local\Temp\ipykernel_49332\2284465309.py:19: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 16. Evaluation — CIDEr & METEOR

In [58]:
import torch, os, json, nltk
from tqdm import tqdm
from torch.cuda.amp import autocast
import evaluate

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

meteor = evaluate.load("meteor")

try:
    from pycocoevalcap.cider.cider import Cider
    cider_scorer = Cider()
    HAS_CIDER = True
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "pycocoevalcap", "-q"])
    from pycocoevalcap.cider.cider import Cider
    cider_scorer = Cider()
    HAS_CIDER = True

# Load best checkpoint
ckpts = sorted([f for f in os.listdir(PATHS["checkpoints"]) if f.endswith(".pt")])
if ckpts:
    ckpt_path = os.path.join(PATHS["checkpoints"], ckpts[-1])
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    print(f"Loaded: {ckpts[-1]}  (train={ckpt['train_loss']:.4f}, val={ckpt['val_loss']:.4f})")

model.eval()
preds, refs = [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Generating"):
        feats    = batch["features"].to(device)
        asr_ids  = batch["asr_input_ids"].to(device)
        asr_mask = batch["asr_attn_mask"].to(device)
        with autocast():
            p = model.generate_caption(feats, asr_ids, asr_mask)
        preds.extend(p)
        refs.extend(batch["captions"])

meteor_score = meteor.compute(predictions=preds, references=refs)["meteor"]
cider_score  = None
if HAS_CIDER:
    pred_d = {i: [p] for i, p in enumerate(preds)}
    ref_d  = {i: [r] for i, r in enumerate(refs)}
    cider_score, _ = cider_scorer.compute_score(ref_d, pred_d)

print(f"\n{'='*45}")
print(f"  METEOR : {meteor_score:.4f}  (target > 0.085)")
if cider_score is not None:
    print(f"  CIDEr  : {cider_score:.4f}  (target > 35)")
print(f"{'='*45}")

# Save examples
examples = [{"ref": refs[i], "pred": preds[i]} for i in range(min(15, len(preds)))]
with open(os.path.join(PATHS["outputs"], "examples.json"), "w") as f:
    json.dump(examples, f, indent=2)
print("Qualitative examples saved.")


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\amans\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\amans\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\amans\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
C:\Users\amans\AppData\Local\Temp\ipykernel_49332\1926792029.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be ex

Loaded: epoch_10.pt  (train=nan, val=3.6306)


C:\Users\amans\AppData\Local\Temp\ipykernel_49332\1926792029.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Generating: 100%|██████████████████████████████████████████████████████████████████████| 34/34 [00:37<00:00,  1.11s/it]



  METEOR : 0.1168  (target > 0.085)
  CIDEr  : 0.2386  (target > 35)
Qualitative examples saved.


## 17. Qualitative Examples

In [60]:
import json, os

with open(os.path.join(PATHS["outputs"], "examples.json")) as f:
    examples = json.load(f)

print(f"{'='*60}")
for i, ex in enumerate(examples[:10]):
    print(f"[{i+1}]")
    print(f"  REF  : {ex['ref']}")
    print(f"  PRED : {ex['pred']}")
    print()


[1]
  REF  : into a cloth add the mashed potatoes and squeeze out all the water
  PRED : put the mash into a bowl

[2]
  REF  : take the cooked mashed potatoes and flour in a bowl
  PRED : add salt and salt to a bowl

[3]
  REF  : add a little salt white pepper stir and add milk in batches to form a batter
  PRED : add salt to a pan

[4]
  REF  : add the starch into the batter and let the batter sit for about 20 minutes
  PRED : put the mash into a bowl

[5]
  REF  : melt little butter and olive oil in a pan
  PRED : add salt to a pan

[6]
  REF  : pour a ladle full of batter and spread it around
  PRED : place the mash in a bowl

[7]
  REF  : let it cook for 2-3 minutes on both side until golden and serve
  PRED : add salt to a pan

[8]
  REF  : mix 2 tbsp of chili garlic sauce with the hoisin and keep it aside
  PRED : add the spring rolls to a pan

[9]
  REF  : take the shrimp and cut it into two halves
  PRED : add the spring rolls to a pan

[10]
  REF  : soften the rice paper by s

## 18. Ablation — Visual-Only vs Visual+ASR

In [62]:
import torch, evaluate
from tqdm import tqdm
from torch.cuda.amp import autocast

meteor = evaluate.load("meteor")

def run_eval(loader, use_asr, tag):
    model.eval()
    preds, refs = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=tag):
            feats = batch["features"].to(device)
            asr_ids  = batch["asr_input_ids"].to(device) if use_asr else None
            asr_mask = batch["asr_attn_mask"].to(device) if use_asr else None
            with autocast():
                p = model.generate_caption(feats, asr_ids, asr_mask)
            preds.extend(p)
            refs.extend(batch["captions"])
    return meteor.compute(predictions=preds, references=refs)["meteor"]

m_with    = run_eval(val_loader, use_asr=True,  tag="Visual+ASR")
m_without = run_eval(val_loader, use_asr=False, tag="Visual-only")

print(f"\n{'='*45}")
print(f"  Visual + ASR  : {m_with:.4f}")
print(f"  Visual only   : {m_without:.4f}")
print(f"  ASR delta     : {m_with - m_without:+.4f}")
print(f"{'='*45}")


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\amans\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\amans\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\amans\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
C:\Users\amans\AppData\Local\Temp\ipykernel_49332\3334184766.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Visual-only: 100%|█████████████████████████████████████████████████████████████████████| 34/34 [03:15<00:00,  5.75s/it]



  Visual + ASR  : 0.1168
  Visual only   : 0.0145
  ASR delta     : +0.1023


In [69]:
# Did CIDEr print above the ablation output?
# If not, run this quick standalone check:
from pycocoevalcap.cider.cider import Cider
pred_d = {i: [p] for i, p in enumerate(preds)}
ref_d  = {i: [r] for i, r in enumerate(refs)}
score, _ = Cider().compute_score(ref_d, pred_d)
print(f"CIDEr: {score:.4f}")

CIDEr: 0.2386


In [71]:
import torch, json, os, subprocess
import cv2, h5py, numpy as np
from torch.cuda.amp import autocast

def infer_video(video_path, model, tokenizer, device,
                feat_extractor, transform,
                max_feat_len=128, feat_dim=1024):
    """
    Given a raw .mp4 file, generate dense captions with timestamps.
    Returns a list of {"start", "end", "caption"} dicts.
    """

    # ── 1. Extract S3D features ───────────────────────────────────────────────
    feats, times = extract_s3d_features(video_path)   # reuses Section 10 function
    duration = times[-1] if len(times) > 0 else 0

    # ── 2. Slide a window across the video ───────────────────────────────────
    #    Use the annotation segment lengths from training as window guidance
    #    Default: 30s windows with 15s stride
    WINDOW  = 30.0
    STRIDE  = 15.0
    windows = []
    start = 0.0
    while start < duration:
        end = min(start + WINDOW, duration)
        windows.append((start, end))
        start += STRIDE

    results = []
    model.eval()

    for (start, end) in windows:
        # Get features for this window
        mask      = (times >= start) & (times <= end)
        seg_feats = feats[mask]

        if len(seg_feats) == 0:
            continue

        # Pad / truncate
        if len(seg_feats) > max_feat_len:
            seg_feats = seg_feats[:max_feat_len]
        else:
            pad = np.zeros((max_feat_len - len(seg_feats), feat_dim), dtype=np.float32)
            seg_feats = np.vstack([seg_feats, pad])

        feat_tensor = torch.tensor(seg_feats).unsqueeze(0).to(device)  # (1, T, D)

        # ── 3. Generate caption ───────────────────────────────────────────────
        with torch.no_grad():
            with autocast():
                caption = model.generate_caption(feat_tensor)[0]

        results.append({
            "start"  : round(start, 1),
            "end"    : round(end,   1),
            "caption": caption,
        })

    return results

In [75]:
import subprocess
download=input("Enter Youtube Video Link:") 
subprocess.run([
    "yt-dlp", "--js-runtimes", "node",
    "-f", "best[height<=360]",
    "-o", r"D:\MS\UMD\Courses\Spring-2026\NLP\data\raw_videos\new_test.mp4",
    download
])

VIDEO_PATH = r"D:\MS\UMD\Courses\Spring-2026\NLP\DVC_Project\data\raw_videos\new_test.mp4"
captions = infer_video(VIDEO_PATH, model, tokenizer, device,
                       feature_extractor, transform)

Enter Youtube Video Link: https://www.youtube.com/watch?v=UhSlMykeesQ&t=994s


C:\Users\amans\AppData\Local\Temp\ipykernel_49332\1900887321.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [ ]:
import torch

# Compile model
print("Compiling model...")
model = torch.compile(model, mode="reduce-overhead")
print("✅ Model compiled")

# TF32 + cuDNN
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
print("✅ TF32 + cuDNN enabled")

# Whisper to GPU
model_whisper = model_whisper.to("cuda")
print("✅ Whisper on CUDA")

print(f"\nGPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.memory_allocated()/1e9:.2f} / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import subprocess, os

url  = input("Enter YouTube cooking video URL: ").strip()
dest = r"D:\MS\UMD\Courses\Spring-2026\NLP\data\raw_videos\demo_test.mp4"

# Remove old file if exists
if os.path.exists(dest):
    os.remove(dest)
    print("Removed previous demo_test.mp4")

print(f"Downloading...")
dl = subprocess.run([
    "yt-dlp", "--js-runtimes", "node",
    "-f", "best[height<=360]",
    "-o", dest,
    "--no-playlist",
    "--quiet",
    url
], capture_output=True, text=True)

if dl.returncode != 0:
    print(f"❌ Download failed:\n{dl.stderr[:300]}")
else:
    size_mb = os.path.getsize(dest) / 1e6
    print(f"✅ Downloaded: demo_test.mp4 ({size_mb:.1f} MB)")

In [ ]:
import time, json
import numpy as np

VIDEO_PATH = r"D:\MS\UMD\Courses\Spring-2026\NLP\DVC_Project\data\raw_videos\demo_test.mp4"

results, transcript = infer_video_full(
    video_path    = VIDEO_PATH,
    model         = model,
    model_whisper = model_whisper,
    tokenizer     = tokenizer,
    device        = device,
    window_sec    = 30.0,   # caption every 30s window
    stride_sec    = 15.0,   # shift window by 15s each step
    batch_size    = 8,      # windows processed simultaneously on GPU
)

In [ ]:
import json, os

# Save
out = {
    "video_url" : url,
    "transcript": transcript,
    "captions"  : results,
}
out_path = os.path.join(PATHS["outputs"], "demo_inference.json")
with open(out_path, "w") as f:
    json.dump(out, f, indent=2)

# Pretty print
print(f"\n{'═'*60}")
print(f"  FINAL CAPTIONS")
print(f"{'═'*60}")
print(f"  {'#':>3}  {'Time':^15}  Caption")
print(f"  {'─'*3}  {'─'*15}  {'─'*35}")
for r in results:
    time_str = f"{r['start']:.0f}s → {r['end']:.0f}s"
    print(f"  {r['segment_id']:>3}  {time_str:^15}  {r['caption']}")

print(f"\n  Saved → {out_path}")
print(f"{'═'*60}")